# SmolVLM2-2.2B-Instruct – POPE Evaluation

Notebook đọc kết quả Inference đã lưu trong checkpoint, kiểm tra dữ liệu, tính các chỉ số đánh giá và xuất bảng tổng hợp cùng ma trận nhầm lẫn.

**Đầu vào:** `/kaggle/working/pope_smollvm2_checkpoint.csv`

**Đầu ra:**
- `evaluation_summary.json`
- `POPE_SmolVLM2_Metrics.csv`
- `confusion_matrix.png`

Notebook này không cần tải lại mô hình SmolVLM2 hoặc chạy inference.

## Bước 1. Import thư viện và khai báo đường dẫn

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

CHECKPOINT_PATH = "/kaggle/working/pope_smollvm2_checkpoint.csv"
SUMMARY_PATH = "/kaggle/working/evaluation_summary.json"
METRICS_CSV_PATH = "/kaggle/working/POPE_SmolVLM2_Metrics.csv"
CONFUSION_MATRIX_PATH = "/kaggle/working/confusion_matrix.png"

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError(
        f"Không tìm thấy file kết quả: {CHECKPOINT_PATH}. "
        "Hãy tải file CSV checkpoint lên Kaggle hoặc chạy Notebook Inference trước."
    )

results_df = pd.read_csv(CHECKPOINT_PATH)
print("Tổng số dòng trong checkpoint:", len(results_df))
display(results_df.head())

## Bước 2. Kiểm tra và lọc kết quả hợp lệ

Chỉ giữ các mẫu có trạng thái `success` hoặc `unknown_response`, nhãn thực tế là `yes/no` và dự đoán là `yes/no/unknown`. Các ID trùng lặp chỉ giữ bản ghi cuối cùng.

In [ ]:
required_columns = {"id", "status", "ground_truth", "prediction"}
missing_columns = required_columns - set(results_df.columns)

if missing_columns:
    raise ValueError(f"Checkpoint thiếu các cột: {sorted(missing_columns)}")

success_df = results_df[
    results_df["status"].astype(str).str.lower().isin(
        ["success", "unknown_response"]
    )
].copy()

success_df = success_df.drop_duplicates(subset="id", keep="last")

success_df["ground_truth"] = success_df["ground_truth"].astype(str).str.lower().str.strip()
success_df["prediction"] = success_df["prediction"].astype(str).str.lower().str.strip()

eval_df = success_df[
    success_df["ground_truth"].isin(["yes", "no"]) &
    success_df["prediction"].isin(["yes", "no", "unknown"])
].copy()

if eval_df.empty:
    raise ValueError("Không có mẫu hợp lệ để đánh giá. Hãy kiểm tra checkpoint.")

print("Số mẫu đủ điều kiện đánh giá:", len(eval_df))
print("Phân bố trạng thái:")
print(results_df["status"].value_counts(dropna=False))
print("Số câu trả lời unknown:", int((eval_df["prediction"] == "unknown").sum()))

## Bước 3. Tính các chỉ số đánh giá

Theo cách xử lý trong notebook gốc, câu trả lời `unknown` được tính là sai bằng cách chuyển thành nhãn đối lập với ground truth. Nhờ đó, mẫu không xác định không bị loại khỏi phép tính chỉ số.

In [ ]:
y_true = eval_df["ground_truth"].copy()
y_pred_original = eval_df["prediction"].copy()

n_unknown = int((y_pred_original == "unknown").sum())

# Quy ước theo notebook gốc: unknown được tính là câu trả lời sai
y_pred = y_pred_original.where(
    y_pred_original != "unknown",
    y_true.map({"yes": "no", "no": "yes"})
)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label="yes", zero_division=0)
recall = recall_score(y_true, y_pred, pos_label="yes", zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label="yes", zero_division=0)

tn, fp, fn, tp = confusion_matrix(
    y_true, y_pred, labels=["no", "yes"]
).ravel()

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0

metrics = {
    "model": "SmolVLM2-2.2B-Instruct",
    "dataset": "POPE",
    "total_rows_in_checkpoint": int(len(results_df)),
    "total_evaluated": int(len(eval_df)),
    "unknown_responses_counted_as_incorrect": n_unknown,
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1_score": float(f1),
    "false_positive_rate": float(fpr),
    "confusion_matrix": {
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp)
    }
}

metrics_df = pd.DataFrame([{
    "Model": metrics["model"],
    "Dataset": metrics["dataset"],
    "Total Samples": metrics["total_evaluated"],
    "Unknown Responses": n_unknown,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1-score": f1,
    "FPR": fpr
}])

display(metrics_df.round(4))

## Bước 4. Lưu bảng chỉ số và file JSON

In [ ]:
with open(SUMMARY_PATH, "w", encoding="utf-8") as f:
    json.dump(metrics, f, ensure_ascii=False, indent=4)

metrics_df.to_csv(METRICS_CSV_PATH, index=False, encoding="utf-8-sig")

print("Đã lưu:", SUMMARY_PATH)
print("Đã lưu:", METRICS_CSV_PATH)

## Bước 5. Tạo và lưu Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=["no", "yes"])

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No", "Yes"],
    yticklabels=["No", "Yes"]
)
plt.xlabel("Predicted")
plt.ylabel("Ground Truth")
plt.title("SmolVLM2 - POPE Confusion Matrix")
plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=300, bbox_inches="tight")
plt.show()

print("Đã lưu:", CONFUSION_MATRIX_PATH)

## Bước 6. Kiểm tra các file đầu ra

Sau khi chạy hết Notebook, tải các file sau từ `/kaggle/working/` để đưa lên thư mục `Output/` trên GitHub:

- `evaluation_summary.json`
- `POPE_SmolVLM2_Metrics.csv`
- `confusion_matrix.png`

Lưu ý: các chỉ số chỉ phản ánh số mẫu hợp lệ thực tế trong checkpoint, không mặc định là 9.000 mẫu.